# 7.18 後訓練有哪些不同教法？


問題是「只回數字：1+2=?」。可以示範3，也可讓人比较3與「答案是3喔！」，或者先作答4再按正誤給0分。先看三種給模型的材料，再记方法名字。

示範提供期望回答；偏好提供同題哪篇较好；回饋提供作答後的評分。這個數字叫reward，不是模型情绪，也不自動等于真實品質；此處只用指定資料卡比较。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
demonstration = {"question": "只回數字：1+2=?", "answer": "3"}
preference = {"question": demonstration["question"], "chosen": "3", "rejected": "答案是3喔！"}
feedback = {"question": demonstration["question"], "sampled_answer": "4", "reward": 0}
print("示範給什麼", demonstration["answer"])
print("偏好比較什麼", preference["chosen"], "勝過", preference["rejected"])
print("作答後得到什麼", feedback["sampled_answer"], feedback["reward"])

輸入是我們人工寫的字典；`chosen`表示被選為較佳、`rejected`表示較差，`sampled_answer`記錄作答，這裡只是用`4`示意，沒有真的抽樣模型。輸出依序是示範`3`、偏好`3`勝過附加句、錯答`4`得到0。偏好兩篇都算對，但後者違反只回數字；回饋則採這個例子的正確性規則。換任務時，不能沿用數字0就假裝已定義完整標準。

理想回答监督是SFT；偏好對直接調回答傾向的一種方法是DPO，需可靠比較與固定參考：此次訓練起點保留、不再更新的模型副本。也可先用人類偏好訓練獎勵模型，再對回答取樣、评分、更新，這條人類回饋流程叫RLHF，PPO是其中一種更新方法；兩名詞層次不同。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-07-18-feedback-materials.svg")))

這些是可選教法，不必依序全跑。若回饋来自程序驗算，不因使用PPO就叫人類回饋。[InstructGPT](https://arxiv.org/pdf/2203.02155v1)和[DeepSeek-R1](https://arxiv.org/pdf/2501.12948v1)給出不同接續配方，方法機制在[第13章](https://birdhackor.github.io/tiny-perceptron-vlm/13.1.html)展開；後訓練也可能改善任務能力，须另用新題查證。

練習把問題改成「用一句話解釋1+2」，先寫出新的示範，並重新判斷原偏好是否還合理。光交換`chosen`和`rejected`不一定足夠：原附加句沒有真的解釋加法。最後說明若用真人比較、程式驗算或另一個模型評分，回饋的來源各是哪一種；方法縮寫不應遮住這個差別。

<details>
<summary>補充：實作約定與原始紀錄</summary>

後訓練能改變的不只語氣。若示範包含解題步驟，或回饋能辨認答案正誤，它也可能改善模型做某些任務的能力；成功與否仍要用新題檢查。另一個實際配方是[DeepSeek-R1論文2.2與2.3節](https://arxiv.org/pdf/2501.12948v1)：一條路線從已完成預訓練的**底座模型**直接用作答回饋做**強化學習，英文縮寫RL**；另一條先用少量理想解題示範，教出清楚的起始回答方式，再接後續訓練。這批起步時加入的示範稱為**冷啟動資料**。兩條路線說明SFT不是所有RL配方不可省略的前一步，不代表我們的微型模型能重現論文能力。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
